In [8]:
# ── cell 1: imports ────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

try:
    from diptest import diptest as _diptest
    HAVE_DIPTEST = True
except Exception:
    HAVE_DIPTEST = False
    print("diptest not installed — dip test skipped. `pip install diptest` to enable.")

warnings.filterwarnings("ignore", category=FutureWarning)

WORLDCOVER = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse veg", 70: "Snow and ice",
    80: "Permanent water", 90: "Herbaceous wetland", 95: "Mangroves",
    100: "Moss and lichen",
}

# reasonable defaults (diagnostics, not tuned optima)
PCA_VAR       = 0.95
K_MAX         = 20
GMM_COV       = "diag"
BIC_REL_STRONG = 0.10
DIP_ALPHA     = 0.05
DIP_PCS       = 3
SIL_MAX_N     = 5000
HOPKINS_N     = 500
MIN_POINTS    = 500
MAX_PER_CLASS = 20000
RANDOM_STATE  = 42

In [9]:
# ── cell 2: diagnostics ────────────────────────────────────────────────────
def hopkins(X, m, rng):
    n, d = X.shape
    m = min(m, n - 1)
    if m < 5:
        return np.nan
    lo, hi = X.min(0), X.max(0)
    rand = rng.uniform(lo, hi, size=(m, d)).astype(np.float32)
    u = NearestNeighbors(n_neighbors=1).fit(X).kneighbors(rand)[0][:, 0]
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    idx = rng.choice(n, m, replace=False)
    w = nn.kneighbors(X[idx])[0][:, 1]
    su, sw = u.sum(), w.sum()
    return float(su / (su + sw)) if (su + sw) > 0 else np.nan

def gmm_bic_sweep(X):
    bics = []
    for k in range(1, K_MAX + 1):
        gm = GaussianMixture(k, covariance_type=GMM_COV,
                             random_state=RANDOM_STATE, n_init=1, max_iter=200).fit(X)
        bics.append(gm.bic(X))
    best_k = int(np.argmin(bics) + 1)
    b1, bbest = bics[0], bics[best_k - 1]
    rel = (b1 - bbest) / abs(b1) if b1 != 0 else 0.0
    return best_k, float(rel)

def dip_pvalues(X, n_pcs, rng):
    if not HAVE_DIPTEST:
        return []
    out = []
    for j in range(min(n_pcs, X.shape[1])):
        x = X[:, j]
        if x.size > 20000:
            x = rng.choice(x, 20000, replace=False)
        try:
            _, p = _diptest(np.ascontiguousarray(x, dtype=np.float64))
            out.append(float(p))
        except Exception:
            pass
    return out

def verdict(rel_bic, best_k, sil, dip_min_p):
    """rel_bic can flag strong cases but its magnitude is unstable in high-dim,
    so it must not solo-veto. best_k>=2 + significant dip + real silhouette =
    overlapping-but-real structure (the built-up-type case)."""
    dip_multi = (dip_min_p is not None) and (dip_min_p < DIP_ALPHA)
    sil_ok    = (sil is not None) and (sil > 0.15)
    sil_weak  = (sil is not None) and (sil > 0.05)
    multi_k   = best_k >= 2
    votes = sum([rel_bic >= BIC_REL_STRONG, multi_k, dip_multi, sil_ok])
    if rel_bic >= BIC_REL_STRONG and multi_k and (sil_ok or dip_multi):
        return "SUBDIVIDES"
    if multi_k and dip_multi and sil_weak:
        return "SUBDIVIDES (overlapping)"
    if votes >= 2:
        return "WEAK / AMBIGUOUS"
    return "UNIMODAL"

def screen_class(X, rng, max_n=MAX_PER_CLASS):
    if X.shape[0] > max_n:
        X = X[rng.choice(X.shape[0], max_n, replace=False)]
    n = X.shape[0]
    Xs = StandardScaler().fit_transform(X)
    Xp = PCA(n_components=PCA_VAR, random_state=RANDOM_STATE).fit_transform(Xs)
    best_k, rel = gmm_bic_sweep(Xp)
    sil = None
    if best_k >= 2:
        lab = GaussianMixture(best_k, covariance_type=GMM_COV,
                              random_state=RANDOM_STATE, n_init=1).fit_predict(Xp)
        if len(np.unique(lab)) >= 2:
            si = rng.choice(n, min(SIL_MAX_N, n), replace=False)
            try:
                sil = float(silhouette_score(Xp[si], lab[si]))
            except Exception:
                sil = None
    dips = dip_pvalues(Xp, DIP_PCS, rng)
    dip_min = min(dips) if dips else None
    return {
        "n_used": n,
        "intrinsic_dim": Xp.shape[1],
        "best_k_bic": best_k,
        "rel_bic_gain": round(rel, 4),
        "silhouette_bestk": round(sil, 3) if sil is not None else np.nan,
        "dip_min_p": round(dip_min, 4) if dip_min is not None else np.nan,
        "hopkins": round(hopkins(Xp, HOPKINS_N, rng), 3),
        "verdict": verdict(rel, best_k, sil, dip_min),
    }

In [10]:
# ── cell 3: stratified pull straight off disk ──────────────────────────────
import duckdb

GLOB      = '../data/raw/**/*.parquet'
LABEL_COL = 'label'
PER_CLASS = 20000

con = duckdb.connect()

# peek at the on-disk schema first
schema = con.execute(f"SELECT * FROM read_parquet('{GLOB}') LIMIT 1").df()
print("on-disk columns:", list(schema.columns))

# stratified sample: <=PER_CLASS rows per class, ALL rows for classes below that
df = con.execute(f"""
    WITH ranked AS (
        SELECT *,
               row_number() OVER (PARTITION BY {LABEL_COL} ORDER BY random()) AS _rn
        FROM read_parquet('{GLOB}')
    )
    SELECT * EXCLUDE (_rn) FROM ranked WHERE _rn <= {PER_CLASS}
""").df()

print(f"\npulled {len(df):,} rows")
print(df[LABEL_COL].value_counts().sort_index())

on-disk columns: ['lat', 'lon', 'label', 'label_name', 'homogeneity', 'h3_cell', 'sample_cell', 'A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


pulled 220,000 rows
label
10     20000
20     20000
30     20000
40     20000
50     20000
60     20000
70     20000
80     20000
90     20000
95     20000
100    20000
Name: count, dtype: int64


In [11]:
# ── cell 4: build X_all, labels_all (auto-detects embedding layout) ─────────
import re

def _is_arraylike(col):
    s = df[col].dropna()
    return (not s.empty) and isinstance(s.iloc[0], (list, tuple, np.ndarray))

arr_cols = [c for c in df.columns if _is_arraylike(c)]

if len(arr_cols) == 1:                                   # single list/array column
    EMB_COL = arr_cols[0]
    print(f"embeddings: list column '{EMB_COL}'")
    mask  = df[EMB_COL].notna().values
    X_all = np.vstack(df.loc[mask, EMB_COL].apply(np.asarray).values).astype(np.float32)
else:                                                    # one column per dimension
    fams = {}
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and c != LABEL_COL:
            m = re.match(r"^([A-Za-z_]*?)_?\d+$", c)
            if m:
                fams.setdefault(m.group(1), []).append(c)
    stem = max(fams, key=lambda k: len(fams[k]))
    EMB_COLS = sorted(fams[stem], key=lambda c: int(re.search(r"(\d+)$", c).group(1)))
    print(f"embeddings: {len(EMB_COLS)} columns {EMB_COLS[0]}..{EMB_COLS[-1]}")
    sub   = df[EMB_COLS]
    mask  = sub.notna().all(axis=1).values
    X_all = sub.loc[mask].to_numpy(np.float32)

labels_all = df[LABEL_COL].to_numpy()[mask]
print(f"\nX_all: {X_all.shape[0]:,} points × {X_all.shape[1]} dims")

embeddings: 64 columns A00..A63

X_all: 220,000 points × 64 dims


In [12]:
# ── cell 5: run the screen -> report DataFrame ─────────────────────────────
rng  = np.random.default_rng(RANDOM_STATE)
rows = []
for code in sorted(pd.unique(labels_all)):
    Xc   = X_all[labels_all == code]
    name = WORLDCOVER.get(int(code), str(code))
    if Xc.shape[0] < MIN_POINTS:
        print(f"· {name:<20} skipped ({Xc.shape[0]} < {MIN_POINTS})")
        continue
    print(f"· {name:<20} {Xc.shape[0]:>10,} pts ...")
    r = screen_class(Xc, rng)
    r |= {"class": name, "code": int(code), "n_total": int(Xc.shape[0])}
    rows.append(r)

report = (pd.DataFrame(rows)
          .set_index("class")
          [["code","n_total","n_used","intrinsic_dim","best_k_bic",
            "rel_bic_gain","silhouette_bestk","dip_min_p","hopkins","verdict"]]
          .sort_values("rel_bic_gain", ascending=False))

· Tree cover               20,000 pts ...
· Shrubland                20,000 pts ...
· Grassland                20,000 pts ...
· Cropland                 20,000 pts ...
· Built-up                 20,000 pts ...
· Bare / sparse veg        20,000 pts ...
· Snow and ice             20,000 pts ...
· Permanent water          20,000 pts ...
· Herbaceous wetland       20,000 pts ...
· Mangroves                20,000 pts ...
· Moss and lichen          20,000 pts ...


In [13]:
# ── cell 6: display ────────────────────────────────────────────────────────
def _color(v):
    return {"SUBDIVIDES":"background-color:#c8e6c9",
            "SUBDIVIDES (overlapping)":"background-color:#dcedc8",
            "WEAK / AMBIGUOUS":"background-color:#fff9c4",
            "UNIMODAL":"background-color:#ffcdd2"}.get(v,"")

report.style.map(_color, subset=["verdict"]).format({
    "n_total":"{:,}","n_used":"{:,}","rel_bic_gain":"{:.3f}",
    "silhouette_bestk":"{:.3f}","dip_min_p":"{:.3f}","hopkins":"{:.3f}"})

,code,n_total,n_used,intrinsic_dim,best_k_bic,rel_bic_gain,silhouette_bestk,dip_min_p,hopkins,verdict
class,,,,,,,,,,
Herbaceous wetland,90,"20,000","20,000",27,20,0.324,0.197,0.000,0.886,SUBDIVIDES
Shrubland,20,"20,000","20,000",26,20,0.295,0.170,0.000,0.868,SUBDIVIDES
Permanent water,80,"20,000","20,000",29,20,0.278,0.093,0.000,0.901,SUBDIVIDES
Snow and ice,70,"20,000","20,000",20,20,0.275,0.144,0.000,0.886,SUBDIVIDES
Bare / sparse veg,60,"20,000","20,000",27,20,0.219,0.110,0.000,0.860,SUBDIVIDES
Grassland,30,"20,000","20,000",33,20,0.201,0.134,0.000,0.834,SUBDIVIDES
Tree cover,10,"20,000","20,000",29,20,0.191,0.110,0.000,0.837,SUBDIVIDES
Mangroves,95,"20,000","20,000",23,20,0.153,0.073,0.000,0.838,SUBDIVIDES
Cropland,40,"20,000","20,000",31,20,0.129,0.115,0.000,0.812,SUBDIVIDES
